# `mart_county_features` — Data Dictionary

The `main_marts.mart_county_features` table is the wide-form output of the emerging-RE feature pipeline.

**Grain:** one row per `(fips, period_end_date)` — where `period_end_date` is Dec-31 of each cohort year.

**Row count:** ~3,200 counties × 15+ cohort years ≈ **48,000+ rows**.

**Columns:** 100 feature columns + 6 targets (grouped below by theme).

**Point-in-time (PIT) correctness:** As of 2026-09-02, ACS-derived features (rental vacancy, rent-to-income, cap rate proxy, renter share, MF permit ratio) are rebuilt PIT-safe. ⚠️ noted below are still latest-per-fips and should be treated as leaky for ML training.

**Auto-generated stats** in section 2 below (null rate, min/max, sample values).

## 0. Setup

In [ ]:
import duckdb, pandas as pd
from pathlib import Path

WAREHOUSE = "/Users/premputtegowda/reps/reps-tracker/emerging_re/data/warehouse/emerging_re.duckdb"
con = duckdb.connect(WAREHOUSE, read_only=True)

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.width', 200)

## 1. Curated data dictionary — grouped by theme

Every column with:
- **Source model** — which `int_*` model produces it
- **Description** — semantic meaning
- **Unit** — share / count / $ / bool / pp / etc.

In [ ]:
DICTIONARY = [
    ('=== IDENTITY ===', None, None, None),
    ('fips', 'int_universe', '5-digit county FIPS code (state+county)', 'text'),
    ('period_end_date', 'int_universe', 'Point-in-time scoring date (always Dec-31 for annual cohorts)', 'date'),
    ('county_name', 'int_universe', 'County name (Census Gazetteer)', 'text'),
    ('state_name', 'int_universe', 'State name (Census Gazetteer)', 'text'),
    ('pop_latest', 'int_census_pop_estimates', 'Latest annual population estimate (Census PEP) as of scoring date', 'count'),
    ('=== MIGRATION ===', None, None, None),
    ('domesticmig_per_1000', 'int_census_pop_estimates', 'Net domestic in-migration per 1,000 residents (annual)', 'per 1,000'),
    ('domesticmig_accel_3yr', 'int_census_pop_estimates', '3-year change in domestic migration rate — is inflow accelerating?', 'per 1,000 Δ'),
    ('internationalmig', 'int_census_pop_estimates', 'Absolute annual international net migration', 'count'),
    ('internationalmig_per_1000', 'int_census_pop_estimates', 'International net migration per 1,000 residents', 'per 1,000'),
    ('irs_net_migration_rate', 'int_irs_migration', 'IRS SOI: net taxpayer migration rate (returns_in − returns_out) / total', 'share'),
    ('irs_inflow_rate', 'int_irs_migration', 'IRS SOI: gross inflow rate of taxpayer returns', 'share'),
    ('irs_income_gap', 'int_irs_migration', 'AGI gap: avg AGI of inflow returns − avg AGI of outflow returns ($)', '$'),
    ('irs_income_gap_pct', 'int_irs_migration', 'Same gap as % of outflow AGI baseline', '%'),
    ('irs_net_agi_per_capita', 'int_irs_migration', 'Net AGI flow per capita — dollar impact of migration', '$/person'),
    ('=== INCOME & WAGES ===', None, None, None),
    ('per_capita_income', 'int_bea_income', 'BEA CAINC1 per-capita income at scoring date (nominal $)', '$'),
    ('pcpi_yoy', 'int_bea_income', 'PCPI YoY change', 'share'),
    ('pcpi_cagr_3yr', 'int_bea_income', '3-year CAGR of per-capita income', 'share/yr'),
    ('pcpi_cagr_5yr', 'int_bea_income', '5-year CAGR of per-capita income (⚠️ may be latest-per-fips, treat as leaky for training)', 'share/yr'),
    ('pcpi_growth_vs_state_1yr_pp', 'int_bea_income', '1yr PCPI growth minus state-mean growth (percentage points)', 'pp'),
    ('pcpi_cagr_3yr_vs_state_pp', 'int_bea_income', '3yr PCPI CAGR minus state-mean CAGR', 'pp'),
    ('wage_growth_yoy', 'int_qcew_growth', 'QCEW avg weekly wage YoY', 'share'),
    ('wage_growth_yoy_demeaned', 'int_qcew_growth', 'Wage growth minus national mean (level shift)', 'share'),
    ('high_wage_share', 'int_qcew_growth', 'Employment share in high-wage sectors (info, finance, prof/tech, mgmt)', 'share'),
    ('high_wage_share_yoy_chg', 'int_qcew_growth', 'YoY change in high-wage employment share', 'share Δ'),
    ('=== EMPLOYMENT ===', None, None, None),
    ('emp_growth_yoy', 'int_qcew_growth', 'QCEW total employment YoY', 'share'),
    ('emp_growth_yoy_demeaned', 'int_qcew_growth', 'Employment growth minus national mean', 'share'),
    ('unemployment_rate', 'int_bls_laus_unemployment', 'BLS LAUS unemployment rate (annual avg)', 'share'),
    ('unemployment_rate_yoy_chg', 'int_bls_laus_unemployment', 'YoY change in unemployment rate', 'share Δ'),
    ('knowledge_emp_yoy', 'int_qcew_knowledge', 'YoY growth in knowledge-sector employment (NAICS 51,54,55)', 'share'),
    ('knowledge_emp_cagr_3yr', 'int_qcew_knowledge', '3yr CAGR of knowledge-sector employment', 'share/yr'),
    ('knowledge_emp_growth_vs_state_pp', 'int_qcew_knowledge', 'Knowledge emp growth minus state mean', 'pp'),
    ('renter_ind_emp_yoy', 'int_mf_features', 'YoY growth in renter-heavy industry employment (retail, food, healthcare, admin support)', 'share'),
    ('renter_ind_emp_cagr_3yr', 'int_mf_features', '3yr CAGR renter-industry employment', 'share/yr'),
    ('renter_ind_emp_growth_vs_state_pp', 'int_mf_features', 'Same, minus state mean', 'pp'),
    ('=== EMPLOYMENT DIVERSITY / ANCHORS ===', None, None, None),
    ('employment_diversity_hhi', 'int_employment_diversity', 'Herfindahl-Hirschman Index of QCEW supersector employment shares. 0 = perfectly diverse, 1 = single-sector monopoly. <0.10 = well-diversified', 'index'),
    ('n_supersectors_reporting', 'int_employment_diversity', 'Count of QCEW supersectors with non-null employment (data-quality indicator; ⚠️ may be latest-per-fips)', 'count'),
    ('max_supersector_share', 'int_employment_diversity', "Largest single supersector's share of employment", 'share'),
    ('anchor_type', 'int_employment_diversity', "Categorical: 'gov'/'edu'/'healthcare'/'tourism'/'none' — dominant institutional anchor", 'categorical'),
    ('anchor_stability_score', 'int_employment_diversity', 'Score 0-1 rating anchor stability (higher = more resilient)', 'score'),
    ('federal_gov_share', 'int_employment_diversity', 'Federal government employment share (military bases, VA hospitals, etc.)', 'share'),
    ('edu_share', 'int_employment_diversity', 'Education services employment share (NAICS 611)', 'share'),
    ('healthcare_share', 'int_employment_diversity', 'Healthcare & social assistance employment share (NAICS 62)', 'share'),
    ('tourism_share', 'int_employment_diversity', 'Accommodation + food services employment share (NAICS 72)', 'share'),
    ('=== DEMOGRAPHICS ===', None, None, None),
    ('age_20_34_share', 'int_acs_demographics', 'Share of population aged 20-34 (renter-heavy cohort) from ACS', 'share'),
    ('age_20_34_share_yoy_chg', 'int_acs_demographics', 'YoY change in 20-34 share', 'share Δ'),
    ('age_20_34_growth_3yr_cagr', 'int_acs_demographics', '3yr CAGR of 20-34 population share', 'share/yr'),
    ('foreign_born_share', 'int_acs_demographics', 'Foreign-born population share (ACS)', 'share'),
    ('foreign_born_share_yoy_chg', 'int_acs_demographics', 'YoY change', 'share Δ'),
    ('foreign_born_cagr_3yr', 'int_acs_demographics', '3yr CAGR of foreign-born share', 'share/yr'),
    ('=== HOUSING SUPPLY & PERMITS ===', None, None, None),
    ('permits_per_1000', 'int_bps_permits', 'Building permits (total units) per 1,000 residents, annual', 'per 1,000'),
    ('permits_per_1000_10yr_baseline', 'int_bps_permits', '10-year trailing mean of permits_per_1000 (baseline)', 'per 1,000'),
    ('permits_intensity_ratio', 'int_bps_permits', 'Current permits_per_1000 / 10yr baseline (>1 = elevated supply)', 'ratio'),
    ('permits_10yr_baseline_years_available', 'int_bps_permits', 'Number of years in the trailing baseline window (data quality)', 'count'),
    ('mf_permit_ratio', 'int_mf_features', "5+ unit permits / total permits (share of new supply that's MF)", 'share'),
    ('mf_permit_ratio_ts', 'int_mf_features', 'Time-series version of mf_permit_ratio (rolling)', 'share'),
    ('mf_permit_ratio_change_3yr', 'int_mf_features', '3yr change in mf_permit_ratio', 'share Δ'),
    ('enp_yoy', 'int_enp_permits', 'Emerging non-permitted (ENP) growth YoY — proxy for informal MF development', 'share'),
    ('enp_cagr_3yr', 'int_enp_permits', '3yr CAGR of ENP', 'share/yr'),
    ('enp_growth_vs_natl_pp', 'int_enp_permits', 'ENP growth minus national mean', 'pp'),
    ('business_applications', 'int_bfs_business_apps', 'Census BFS business applications count (annual)', 'count'),
    ('business_apps_per_1000', 'int_bfs_business_apps', 'Business applications per 1,000 residents', 'per 1,000'),
    ('business_apps_per_1000_3yr_avg', 'int_bfs_business_apps', '3yr rolling avg of business apps per 1,000', 'per 1,000'),
    ('=== HOME PRICES & RENTS ===', None, None, None),
    ('hpi_yoy', 'int_hpi_features', 'FHFA HPI YoY change (all-transactions)', 'share'),
    ('zori_yoy', 'int_zori_features', 'Zillow Observed Rent Index YoY change', 'share'),
    ('hpi_trailing_24mo', 'int_hpi_features', 'HPI cumulative return over trailing 24 months', 'share'),
    ('hpi_5yr_cagr_deficit', 'int_hpi_features', 'Current 5yr HPI CAGR minus long-run avg — negative = underperformance', 'pp'),
    ('zori_5yr_cagr_deficit', 'int_zori_features', 'Current 5yr ZORI CAGR minus long-run avg', 'pp'),
    ('rent_minus_price_growth', 'mart_county_features', 'zori_yoy − hpi_yoy (positive = rents growing faster than prices → cap expansion)', 'pp'),
    ('rent_growth_vs_wage_growth', 'mart_county_features', 'zori_yoy − wage_growth_yoy (positive = rent burden increasing)', 'pp'),
    ('=== MF & RENTAL MARKET ===', None, None, None),
    ('renter_share', 'int_mf_features', 'Share of occupied housing units that are renter-occupied (ACS)', 'share'),
    ('rental_vacancy_rate', 'int_acs_features', 'Rental vacancy rate (ACS)', 'share'),
    ('rental_vacancy_change_3yr', 'int_mf_features', '3yr change in rental vacancy rate (negative = tightening)', 'share Δ'),
    ('rent_to_income_ratio', 'int_acs_features', 'Median gross rent × 12 / median household income', 'ratio'),
    ('cap_rate_proxy', 'int_acs_features', 'ACS proxy cap rate: median_gross_rent × 12 / median_home_value (SFR+MF blended, includes utilities)', 'share'),
    ('cap_rate_now_ts', 'int_mf_features', 'Time-series version of cap rate proxy', 'share'),
    ('mf_cap_rate_estimated', 'int_mf_features', 'MF-adjusted cap: (HUD FMR 2BR × 12 / median_home_value) + 1.5% MF premium — closer to acquisition caps', 'share'),
    ('mf_cap_rate_change_3yr', 'int_mf_features', '3yr change in mf_cap_rate_estimated', 'share Δ'),
    ('fmr_2br', 'int_hud_fmr', 'HUD Fair Market Rent, 2-bedroom (40th percentile gross rent)', '$/mo'),
    ('fmr_2br_yoy', 'int_hud_fmr', 'FMR 2BR YoY change', 'share'),
    ('fmr_2br_cagr_3yr', 'int_hud_fmr', 'FMR 2BR 3yr CAGR', 'share/yr'),
    ('fmr_2br_cagr_3yr_vs_state_pp', 'int_hud_fmr', 'FMR CAGR minus state-mean CAGR', 'pp'),
    ('vt_vacancy_level', 'int_zip_vacancy', 'USPS ZIP-level vacancy rate (Valassis) — often null (HUD blocked)', 'share'),
    ('vt_vacancy_change_3yr', 'int_zip_vacancy', '3yr change in vacancy level', 'share Δ'),
    ('vt_vacancy_change_5yr', 'int_zip_vacancy', '5yr change in vacancy level', 'share Δ'),
    ('=== DISQUALIFIERS (boolean gates) ===', None, None, None),
    ('disq_aging_trap', 'mart_county_features', 'TRUE if 20-34 share is declining AND foreign_born share is declining (demographic decay)', 'bool'),
    ('disq_sector_concentration', 'mart_county_features', 'TRUE if max_supersector_share > threshold (single-industry town)', 'bool'),
    ('disq_price_ran_25pct', 'mart_county_features', 'TRUE if HPI already ran +25% in trailing 24mo (top-tick risk)', 'bool'),
    ('any_disqualifier', 'mart_county_features', 'TRUE if ANY disq_* flag is set — top-level filter', 'bool'),
    ('=== FORWARD RETURNS & ML TARGETS ===', None, None, None),
    ('hpi_forward_return_24mo', 'mart_county_features', 'Realized 24mo HPI return from period_end_date (LEAKS if used as feature)', 'share'),
    ('hpi_forward_return_36mo', 'mart_county_features', 'Realized 36mo HPI return', 'share'),
    ('hpi_forward_return_60mo', 'mart_county_features', 'Realized 60mo HPI return', 'share'),
    ('zori_forward_return_24mo', 'mart_county_features', 'Realized 24mo ZORI rent return', 'share'),
    ('zori_forward_return_36mo', 'mart_county_features', 'Realized 36mo ZORI rent return', 'share'),
    ('zori_forward_return_60mo', 'mart_county_features', 'Realized 60mo ZORI rent return', 'share'),
    ('target_hpi_24mo_demeaned', 'mart_county_features', 'hpi_forward_return_24mo minus cohort-year national mean — ML training target', 'share'),
    ('target_hpi_36mo_demeaned', 'mart_county_features', '36mo demeaned HPI target', 'share'),
    ('target_hpi_60mo_demeaned', 'mart_county_features', '60mo demeaned HPI target', 'share'),
    ('target_zori_24mo_demeaned', 'mart_county_features', '24mo demeaned ZORI target', 'share'),
    ('target_zori_36mo_demeaned', 'mart_county_features', '36mo demeaned ZORI target — primary ML target used by analog_search', 'share'),
    ('target_zori_60mo_demeaned', 'mart_county_features', '60mo demeaned ZORI target', 'share'),
]

rows = []
current_section = None
for col, src, desc, unit in DICTIONARY:
    if col.startswith('==='):
        current_section = col.strip('= ').strip()
        continue
    rows.append({
        'section': current_section, 'column': col, 'source_model': src,
        'unit': unit, 'description': desc,
    })
dd = pd.DataFrame(rows)
print(f"Total curated columns: {len(dd)}")
dd

## 2. Auto-generated stats — verify coverage per column

For each column: dtype, non-null count, null%, min/max/mean (numeric), sample values (categorical).

In [ ]:
# Column types + non-null counts
schema = con.execute('DESCRIBE main_marts.mart_county_features').df()[['column_name','column_type']]

# Row count + null counts per column
n_rows = con.execute('SELECT COUNT(*) FROM main_marts.mart_county_features').fetchone()[0]

null_query = "SELECT " + ", ".join(f'SUM(CASE WHEN "{c}" IS NULL THEN 1 ELSE 0 END) AS "{c}"' for c in schema['column_name']) + " FROM main_marts.mart_county_features"
nulls = con.execute(null_query).df().T.reset_index()
nulls.columns = ['column_name','n_null']

stats = schema.merge(nulls, on='column_name')
stats['n_non_null'] = n_rows - stats['n_null']
stats['null_pct'] = (stats['n_null'] / n_rows * 100).round(1)
print(f"Total rows in mart: {n_rows:,}")
stats[['column_name','column_type','n_non_null','null_pct']]

In [ ]:
# Numeric distributions — min/median/max/mean for numeric columns
numeric_cols = schema[schema['column_type'].isin(['DOUBLE','BIGINT','INTEGER'])]['column_name'].tolist()
parts = []
for c in numeric_cols:
    parts.append(f'MIN("{c}") AS "{c}__min", '
                 f'PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY "{c}") AS "{c}__p50", '
                 f'MAX("{c}") AS "{c}__max", '
                 f'AVG("{c}") AS "{c}__mean"')
q = "SELECT " + ", ".join(parts) + " FROM main_marts.mart_county_features"
r = con.execute(q).df()

# Reshape to one row per column
records = []
for c in numeric_cols:
    records.append({'column': c, 'min': r[f'{c}__min'][0], 'p50': r[f'{c}__p50'][0],
                    'max': r[f'{c}__max'][0], 'mean': r[f'{c}__mean'][0]})
num_dist = pd.DataFrame(records)
num_dist

In [ ]:
# Categorical sample values — for VARCHAR + BOOLEAN columns
cat_cols = schema[schema['column_type'].isin(['VARCHAR','BOOLEAN'])]['column_name'].tolist()
cat_samples = []
for c in cat_cols:
    if c in ('fips','county_name','state_name','period_end_date'):
        continue  # skip identity columns
    top = con.execute(f'SELECT "{c}", COUNT(*) AS n FROM main_marts.mart_county_features GROUP BY 1 ORDER BY n DESC LIMIT 5').df()
    cat_samples.append({'column': c, 'top_5_values': top.to_dict(orient='records')})
pd.DataFrame(cat_samples)

## 3. Merged view — dictionary + stats + coverage side-by-side

In [ ]:
merged = dd.merge(stats[['column_name','column_type','null_pct']],
                  left_on='column', right_on='column_name', how='left').drop(columns=['column_name'])
merged = merged[['section','column','column_type','null_pct','unit','source_model','description']]
merged

## 4. Reference — source models

The mart is assembled by joining these `int_*` models on `(fips, period_end_date)`:

| Intermediate model | What it produces | Source ingest |
|---|---|---|
| `int_universe` | Counties passing pop + coverage gates | `stg_census_pop_estimates`, `stg_fhfa_hpi`, `stg_zillow_zori` |
| `int_hpi_features` | HPI YoY, 5yr CAGR, trailing returns | `stg_fhfa_hpi` |
| `int_zori_features` | ZORI YoY, 5yr CAGR deficit | `stg_zillow_zori` |
| `int_acs_features` | Rent, income, vacancy, cap proxy (PIT) | `stg_census_acs`, `stg_census_acs_1yr` |
| `int_census_pop_estimates` | Pop, migration, demographics | `stg_census_pep_char`, `stg_census_pep_components` |
| `int_qcew_growth` | Employment, wages, high-wage share | `stg_bls_qcew` |
| `int_bps_permits` | Total permits + intensity ratio | `stg_census_bps` |
| `int_bea_income` | PCPI + growth rates | `stg_bea_cainc1` |
| `int_employment_diversity` | HHI + anchor type + sector shares | `stg_bls_qcew` (supersector) |
| `int_bfs_business_apps` | Business formation | `stg_census_bfs` |
| `int_bls_laus_unemployment` | Unemployment rate + Δ | `stg_bls_laus` |
| `int_mf_features` | MF cap, renter industry, ratios (PIT) | multiple |
| `int_mf_inventory` | Est MF building counts + age (PIT) | `stg_census_acs` B25024/B25034 |
| `int_irs_migration` | Taxpayer flows + income gap | `stg_irs_soi_migration` |
| `int_hud_fmr` | FMR 2BR + growth | `stg_hud_fmr` |
| `int_zip_vacancy` | ZIP-level vacancy (mostly null) | HUD USPS (blocked) |
| `int_enp_permits` | Non-permitted development proxy | `stg_census_enp` |
| `int_acs_demographics` | Age + foreign-born shares | `stg_census_acs` |
| `int_qcew_knowledge` | Knowledge-sector employment | `stg_bls_qcew` |
| `int_national_macro_regime` | National-level regime tag (joined at run-time) | national FRED series |

## 5. PIT-correctness notes

The intermediate models below emit **one row per (fips, scoring_period)** with as-of-aware source lookups (rewritten 2026-09-02). Safe to use as ML inputs:

- `int_acs_features`: rental_vacancy_rate, rent_to_income_ratio, cap_rate_proxy
- `int_mf_features`: renter_share, mf_permit_ratio, rental_vacancy_change_3yr, mf_cap_rate_estimated
- `int_mf_inventory`: MF building estimates + age

**⚠️ NOT rebuilt PIT-safe** (treat as leaky for training):
- `pcpi_cagr_5yr` — from `int_bea_income`, still latest-per-fips
- `n_supersectors_reporting` — from `int_employment_diversity`, likely latest-per-fips